In [ ]:

# # Experimento RKO — Comitê de Agrupamentos (v3)
#
# Usa os 30 comitês independentes por (base, estratégia) gerados em all_results.pkl.
# Cada (base, estratégia, comitê) é resolvido por BRKGA, VNS e ILS, todos sobre o MESMO comitê.
#
# Saídas: ../results/resultados_rko.csv  (métricas + seed de cada execução)
#         ../results/labels_rko.csv      (labels finais fase1/fase2)
#         ../results/erros_rko.csv       (execuções que falharam; são refeitas ao retomar)
#         ../results/config_rko.json     (configuração + RKO_MASTER_SEED)

# %%
import sys
sys.path.append('.')
sys.path.append('../src/clustering')

import os
import json
import time
import zlib
import random
import pickle
import traceback
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import StandardScaler

from ca import ClusteringEnsemble
from particoes_base_e_matriz_de_coassociacao import (
    NAO_HIERARQUICOS,
    HIERARQUICOS,
    _definir_seed_r,
)
from rko import RKO

# ============================================================
# SEED MASTER DO EXPERIMENTO RKO (única seed manual deste notebook)
# ============================================================
RKO_MASTER_SEED = 42   # independente da MASTER_SEED dos comitês

def _id(nome: str) -> int:
    return zlib.crc32(nome.encode("utf-8"))

TAG_RKO      = _id("rko_consenso")
TAG_BASELINE = _id("baseline")

def seed_rko(dataset, estrategia, metodo, comite) -> int:
    """Seed de 32 bits da execução (base, estratégia, método, comitê). Independe da ordem."""
    ss = np.random.SeedSequence(
        entropy=RKO_MASTER_SEED,
        spawn_key=(TAG_RKO, _id(dataset), _id(estrategia), _id(metodo), int(comite)),
    )
    return int(ss.generate_state(1, dtype=np.uint32)[0])

def seed_baseline(dataset, algoritmo) -> int:
    ss = np.random.SeedSequence(
        entropy=RKO_MASTER_SEED,
        spawn_key=(TAG_BASELINE, _id(dataset), _id(algoritmo)),
    )
    return int(ss.generate_state(1, dtype=np.uint32)[0])

def nmi_max(gt, labels):
    return normalized_mutual_info_score(gt, labels, average_method="max")

# ============================================================
# Carregamento dos dados (pkl grande: mantém só a DCO)
# ============================================================
PKL_PATH = '../results/all_results.pkl'

print("Carregando pkl (pode demorar e usar vários GB de RAM)...", flush=True)
with open(PKL_PATH, 'rb') as f:
    dados = pickle.load(f)

all_results  = dados['all_results']
n_clusters   = dados['n_clusters']
ground_truth = dados['ground_truth']
datasets     = dados['datasets']
meta_pkl     = dados['metadata']

N_COMITES = meta_pkl['config']['n_comites']

# libera a coassociação (o RKO só usa a dissociação)
for d in all_results:
    for e in all_results[d]:
        for com in all_results[d][e]['committees'].values():
            com.pop('coassociation', None)

# exige comitês completos
faltando = [(d, e, c) for d in all_results for e in all_results[d]
            for c in range(1, N_COMITES + 1)
            if c not in all_results[d][e]['committees']]
if faltando:
    raise RuntimeError(f"{len(faltando)} comitês ausentes no pkl (ex.: {faltando[:3]}). "
                       "Termine a geração antes do experimento.")

print(f"Datasets: {len(all_results)} | comitês por combinação: {N_COMITES} "
      f"| MASTER_SEED dos comitês: {meta_pkl['config']['master_seed']}")

# ============================================================
# Configurações
# ============================================================
MAX_TIME    = 90
ESTRATEGIAS = ['hierarquica', 'nao_hierarquica', 'mistura']
DATASETS    = list(all_results.keys())
METODOS     = [('BRKGA', {'brkga': 1}), ('VNS', {'vns': 1}), ('ILS', {'ils': 1})]

RESULTADOS_CSV = '../results/resultados_rko.csv'
LABELS_CSV     = '../results/labels_rko.csv'
ERROS_CSV      = '../results/erros_rko.csv'
CONFIG_JSON    = '../results/config_rko.json'

# ------------------------------------------------------------
# Compatibilidade da configuração (não mistura execuções incompatíveis)
# ------------------------------------------------------------
config_rko = {
    "rko_master_seed": RKO_MASTER_SEED,
    "comites_master_seed": meta_pkl['config']['master_seed'],
    "pkl_criado_em": meta_pkl.get('criado_em'),
    "max_time": MAX_TIME,
    "n_comites": N_COMITES,
    "estrategias": ESTRATEGIAS,
    "metodos": [m for m, _ in METODOS],
    "datasets": DATASETS,
    "esquema_seed": "SeedSequence(entropy=RKO_MASTER_SEED, spawn_key=(crc32('rko_consenso'), "
                    "crc32(base), crc32(estrategia), crc32(metodo), comite))",
}
os.makedirs('../results', exist_ok=True)
if os.path.exists(CONFIG_JSON):
    with open(CONFIG_JSON, encoding='utf-8') as f:
        antiga = json.load(f)
    dif = [k for k in config_rko if antiga.get(k) != config_rko[k]]
    if dif:
        raise RuntimeError(f"Configuração incompatível com a execução anterior: {dif}. "
                           "Use outros arquivos *_v3 (ou apague os antigos de propósito).")
else:
    with open(CONFIG_JSON, 'w', encoding='utf-8') as f:
        json.dump(config_rko, f, indent=2, ensure_ascii=False)

print(f'RKO_MASTER_SEED : {RKO_MASTER_SEED}')
print(f'Tempo/heurística: {MAX_TIME}s | Estratégias: {ESTRATEGIAS}')

# ============================================================
# CSV incremental com checkpoint
# ============================================================
CSV_COLUMNS = ["dataset", "estrategia", "metodo", "fase", "comite",
               "silhueta", "ARI", "NMI", "tempo", "seed"]
LABELS_COLUMNS = ["dataset", "estrategia", "metodo", "fase", "comite", "labels"]
ERROS_COLUMNS = ["dataset", "estrategia", "metodo", "comite", "seed", "erro"]


def _init_csv(path, columns):
    if not os.path.exists(path):
        pd.DataFrame(columns=columns).to_csv(path, index=False)


def _append_rows(path, rows, columns):
    with open(path, 'a', newline='', encoding='utf-8') as f:
        pd.DataFrame(rows, columns=columns).to_csv(f, header=False, index=False)
        f.flush()
        os.fsync(f.fileno())


def _load_done(path):
    done_baseline, done_rko = set(), set()
    if not os.path.exists(path):
        return done_baseline, done_rko
    df = pd.read_csv(path)
    if df.empty:
        return done_baseline, done_rko
    df = df.drop_duplicates(subset=["dataset", "estrategia", "metodo", "fase", "comite"], keep="last")
    df = df[df["ARI"].notna()]                       # execuções com erro serão refeitas

    base = df[df["fase"] == "baseline"]
    done_baseline = set(zip(base["dataset"], base["metodo"]))

    rko = df[df["fase"].isin(["fase1", "fase2"])]
    cont = rko.groupby(["dataset", "estrategia", "metodo", "comite"]).size()
    done_rko = {(a, b, c, int(d)) for a, b, c, d in cont[cont == 2].index}
    return done_baseline, done_rko


_init_csv(RESULTADOS_CSV, CSV_COLUMNS)
_init_csv(LABELS_CSV, LABELS_COLUMNS)
_init_csv(ERROS_CSV, ERROS_COLUMNS)
done_baseline, done_rko = _load_done(RESULTADOS_CSV)
print(f'Baselines já concluídos     : {len(done_baseline)}')
print(f'Execuções RKO já concluídas : {len(done_rko)}')

# ============================================================
# Baseline — 14 algoritmos com k verdadeiro (uma vez por dataset),
# com seed derivada para os estocásticos
# ============================================================
ALGORITMOS_BASE = NAO_HIERARQUICOS + HIERARQUICOS


def avaliar_baseline_algoritmos(dataset, padronizar=True):
    X  = datasets[dataset]
    gt = ground_truth[dataset]
    k_real = len(np.unique(gt))
    X_proc = StandardScaler().fit_transform(X) if padronizar else X.copy()

    resultados = []
    for item in ALGORITMOS_BASE:
        nome = item["nome"]
        seed = seed_baseline(dataset, nome) & 0x7FFFFFFF if item["estocastico"] else None
        t0 = time.time()
        try:
            if seed is not None:
                _definir_seed_r(seed)
                np.random.seed(seed)
            labels = item["func"](X_proc, k_real, seed=seed)
            resultados.append({"algoritmo": nome,
                               "ARI": adjusted_rand_score(gt, labels),
                               "NMI": nmi_max(gt, labels),
                               "tempo": round(time.time() - t0, 4),
                               "seed": -1 if seed is None else seed})
        except Exception:
            resultados.append({"algoritmo": nome, "ARI": np.nan, "NMI": np.nan,
                               "tempo": np.nan, "seed": -1 if seed is None else seed})
    return pd.DataFrame(resultados)

# ============================================================
# RKO — Fase 1 (decoder) x Fase 2 (procedimento iterativo)
# ============================================================

def rodar_um_metodo(D, k, gt, instance_name, max_time, metodo, kwargs):
    env = ClusteringEnsemble(D, k=k, instance_name=instance_name, max_time=max_time)
    solver = RKO(env)

    t0 = time.time()
    cost, keys, t = solver.solve(time_total=max_time, **kwargs)
    wall = time.time() - t0

    labels_fase1, medoids = env.decoder(keys)
    sil_fase1 = -env.cost((labels_fase1, medoids))
    ari_fase1 = adjusted_rand_score(gt, labels_fase1)
    nmi_fase1 = nmi_max(gt, labels_fase1)

    labels_fase2 = env.procedimento_iterativo(medoids, verbose=False)
    sil_fase2 = -env.cost((labels_fase2, medoids))
    ari_fase2 = adjusted_rand_score(gt, labels_fase2)
    nmi_fase2 = nmi_max(gt, labels_fase2)

    return {
        'tempo': round(t, 2), 'tempo_real_s': round(wall, 2),
        'silhueta_fase1': round(sil_fase1, 6), 'ARI_fase1': round(ari_fase1, 6), 'NMI_fase1': round(nmi_fase1, 6),
        'silhueta_fase2': round(sil_fase2, 6), 'ARI_fase2': round(ari_fase2, 6), 'NMI_fase2': round(nmi_fase2, 6),
        'labels_fase1': np.asarray(labels_fase1),
        'labels_fase2': np.asarray(labels_fase2),
    }

# ============================================================
# Loop principal: base -> estratégia -> comitê -> método
# ============================================================
total_baseline = len(DATASETS) * len(ALGORITMOS_BASE)
total_rko = len(DATASETS) * len(ESTRATEGIAS) * N_COMITES * len(METODOS)
print(f'\nBaselines: {total_baseline} | Execuções RKO: {total_rko} (~{total_rko * MAX_TIME / 3600:.0f}h no pior caso)')

contador = len(done_rko)

for dataset in DATASETS:
    print(f'\n{"-"*60}\nDataset: {dataset} | k={n_clusters[dataset]} | n={len(ground_truth[dataset])}\n{"-"*60}')
    gt = ground_truth[dataset]
    k  = n_clusters[dataset]

    # ---- baselines (pula se todos já feitos) ----
    if not all((dataset, it["nome"]) in done_baseline for it in ALGORITMOS_BASE):
        baselines = avaliar_baseline_algoritmos(dataset)
        rows_baseline = []
        for _, row in baselines.iterrows():
            if (dataset, row["algoritmo"]) in done_baseline:
                continue
            print(f"    [baseline] {row['algoritmo']:22s} ARI={row['ARI']:.4f}  NMI={row['NMI']:.4f}")
            rows_baseline.append({
                "dataset": dataset, "estrategia": "baseline", "metodo": row["algoritmo"],
                "fase": "baseline", "comite": 0, "silhueta": np.nan,
                "ARI": row["ARI"], "NMI": row["NMI"], "tempo": row["tempo"], "seed": int(row["seed"]),
            })
        if rows_baseline:
            _append_rows(RESULTADOS_CSV, rows_baseline, CSV_COLUMNS)

    # ---- RKO ----
    for estrategia in ESTRATEGIAS:
        print(f'\n  Estratégia: {estrategia}')
        comites = all_results[dataset][estrategia]['committees']

        for comite in range(1, N_COMITES + 1):
            D = comites[comite]['dissociation']

            for metodo, kwargs in METODOS:
                if (dataset, estrategia, metodo, comite) in done_rko:
                    continue

                seed = seed_rko(dataset, estrategia, metodo, comite)
                np.random.seed(seed)
                random.seed(seed)

                instance_name = f"{dataset}_{estrategia}_{metodo}_c{comite}"
                try:
                    res = rodar_um_metodo(D, k, gt, instance_name, MAX_TIME, metodo, kwargs)
                except Exception as e:
                    print(f"    [ERRO] {instance_name}: {e}")
                    _append_rows(ERROS_CSV, [{"dataset": dataset, "estrategia": estrategia,
                                              "metodo": metodo, "comite": comite, "seed": seed,
                                              "erro": traceback.format_exc().replace("\n", " | ")}],
                                 ERROS_COLUMNS)
                    continue   # não grava nada: será refeito ao retomar

                contador += 1
                print(f"    [{contador}/{total_rko}] [{metodo:5s} c={comite:02d}] "
                      f"F1: Sil={res['silhueta_fase1']:.4f} ARI={res['ARI_fase1']:.4f} NMI={res['NMI_fase1']:.4f} | "
                      f"F2: Sil={res['silhueta_fase2']:.4f} ARI={res['ARI_fase2']:.4f} NMI={res['NMI_fase2']:.4f} "
                      f"t={res['tempo']}s")

                # labels primeiro; resultados por último (define "concluído")
                _append_rows(LABELS_CSV, [
                    {"dataset": dataset, "estrategia": estrategia, "metodo": metodo,
                     "fase": "fase1", "comite": comite,
                     "labels": " ".join(map(str, res['labels_fase1'].tolist()))},
                    {"dataset": dataset, "estrategia": estrategia, "metodo": metodo,
                     "fase": "fase2", "comite": comite,
                     "labels": " ".join(map(str, res['labels_fase2'].tolist()))},
                ], LABELS_COLUMNS)

                _append_rows(RESULTADOS_CSV, [
                    {"dataset": dataset, "estrategia": estrategia, "metodo": metodo,
                     "fase": "fase1", "comite": comite,
                     "silhueta": res["silhueta_fase1"], "ARI": res["ARI_fase1"], "NMI": res["NMI_fase1"],
                     "tempo": res["tempo"], "seed": seed},
                    {"dataset": dataset, "estrategia": estrategia, "metodo": metodo,
                     "fase": "fase2", "comite": comite,
                     "silhueta": res["silhueta_fase2"], "ARI": res["ARI_fase2"], "NMI": res["NMI_fase2"],
                     "tempo": res["tempo"], "seed": seed},
                ], CSV_COLUMNS)

print(f'\nResultados: {RESULTADOS_CSV}\nLabels: {LABELS_CSV}\nErros: {ERROS_CSV}')

# %% [markdown]
# ## Conferência final

# %%
df = pd.read_csv(RESULTADOS_CSV).drop_duplicates(
    subset=["dataset", "estrategia", "metodo", "fase", "comite"], keep="last")
print(df.shape)
print(df.groupby("fase").size())
print(df[df["fase"].isin(["fase1", "fase2"])]
      .groupby(["dataset", "estrategia", "metodo", "fase"]).size().describe())  # deve ser 30 por grupo
print("Erros pendentes:", len(pd.read_csv(ERROS_CSV)))
df.head()

